# CME Futures: Portfolio Allocation

The baseline stage ranks complete configurations by equal-weight validation backtest Sharpe. For
each label, this notebook retains the strongest checkpoint and signal concentration for each of
the configured number of distinct model configurations, then evaluates the declared alternative
position sizing methods. Equal weight is not among them: it is the baseline itself, and because
`stage` is not part of `backtest_hash`, running it again here produces a row hashing
identically to its baseline parent, so one of the two is silently lost. Measured in this
case study's own pre-rebuild store: 48 rows stamped `stage='signal'` while carrying
`allocation.method='equal_weight'`, and no allocation-stage equal-weight rows at all.

All allocator lookbacks come from the case-study configuration. The official population is fixed
before execution; machine speed and caught failures cannot change which allocators run.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：投資組合配置

基準階段依等權重驗證期回測的夏普比率，為完整設定排序。對每個標籤，本 Notebook 會依設定指定的不同模型設定數量，為每項設定保留表現最好的檢查點與訊號集中度，接著評估聲明的其他部位規模決定方法。等權重不在其中：它就是基準本身。由於 `stage` 不屬於 `backtest_hash`，在此重新執行會產生與其基準父項相同的資料列雜湊值，導致兩者之一悄悄遺失。本案例研究重建前的資料庫中，實際有 48 列標記為 `stage='signal'`，卻帶著 `allocation.method='equal_weight'`，而配置階段的等權重資料列一列也沒有。

所有配置方法的回溯期間都取自案例研究的設定。官方群體在執行前即已固定；機器速度與被捕捉的失敗都無法改變要執行哪些配置方法。
<!-- ML4T_ZH_END c0000-p000 -->


## What allocation decides, and why it is not a detail

The baseline held every selected product in the same size. That is a real choice, not the
absence of one, and it says something specific: that the signal's ranking carries information
about *which* products to hold and nothing about *how much* of each.

An allocator disputes that. It sizes positions by some property the ranking does not capture -
how volatile a product has been, how it co-moves with the others, how confident the model was.
The claim is that two products the signal ranks equally are not equally worth the same dollar
risk.

For futures this is a larger effect than it would be in equities, and the reason is in the
instruments. A gold contract and a natural-gas contract with equal notional exposure carry
entirely different risk, because their volatilities differ by a wide margin and they do not
move together. An equal-weight book of futures is therefore not a neutral book: it is one
whose realized risk is dominated by whichever contracts happen to be most volatile, and its
overall behaviour can be driven by two or three positions out of thirty regardless of what the
signal said.

### What the alternatives are actually doing


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 配置決定什麼，為何它不是細節

基準讓每個選中的商品持有相同規模的部位。這是實際的選擇，而非沒有做選擇；它傳達一項具體主張：訊號排序只包含*該持有哪些*商品的資訊，不包含*各該持有多少*的資訊。

配置方法對此提出異議。它會根據排序未捕捉到的特性決定部位規模，例如商品過去的波動程度、與其他商品共同波動的情況，或模型的信心程度。它主張，即使兩項商品的訊號排名相同，也不代表它們值得承擔相同金額的風險。

對期貨而言，這種影響比股票更大，原因在於工具本身。名目曝險相同的黃金合約與天然氣合約，承擔的風險截然不同，因為兩者的波動率差距很大，也不會同步波動。因此，等權重期貨投資組合並非中性組合：其已實現風險由碰巧波動最大的合約主導；即使訊號另有指示，三十個部位中的兩、三個仍可能左右整體表現。

### 各種替代方法實際上在做什麼
<!-- ML4T_ZH_END c0001-p000 -->

The declared methods differ in how much they estimate, and that is the axis to read them on.
Sizing inversely to a product's own volatility uses one number per product and no relationship
between them - it equalizes risk contribution under the assumption that the products are
independent, which they are not, but it is robust because a single volatility is estimated
accurately from little data. Methods that use the covariance between products can in principle
do better, because diversification is a property of the relationships rather than of any one
series. In practice they must estimate far more quantities from the same history, and a
covariance matrix estimated from a short window is dominated by noise that the optimizer then
treats as signal - which is why the more sophisticated method is not reliably the better one
and why they are compared here rather than assumed.

### Why the lookbacks are declared in configuration

Every allocator reads a history to estimate from, and the length of that history is a free
parameter that materially changes the result: a short window tracks a regime change quickly
and is noisy, a long one is stable and stale. Choosing it by validation Sharpe would be fitting
the allocator to the same path it is then assessed on, and the improvement would be
indistinguishable from a real one. The lookbacks come from `config/setup.yaml` for the same
reason the risk-overlay parameters do.

### Why equal weight is excluded here rather than re-run


<!-- ML4T_ZH_BEGIN c0001-p001 -->
聲明的各種方法，差異在於需要估計多少資訊；應沿著這個面向解讀。依商品自身波動率的倒數決定部位規模，每項商品只需估計一個數字，不考慮商品彼此的關係。它假設商品相互獨立，藉此使風險貢獻相等；雖然商品並不獨立，但少量資料就能準確估計單一波動率，因此此法相當穩健。使用商品間共變異數的方法，原則上可以做得更好，因為分散效果取決於彼此關係，而非任何單一序列。但實務上，它們必須從相同的歷史資料估計更多數量；短期視窗估計的共變異數矩陣容易受雜訊主導，最佳化器又會把雜訊當成訊號。因此，較複雜的方法不一定較好，這也是此處要實際比較，而非預設其優劣的原因。

### 為何在設定中聲明回溯期間

每種配置方法都會讀取一段歷史資料來估計參數，而這段歷史的長度是會顯著改變結果的自由參數：短視窗能快速追蹤市場狀態的變化，卻較嘈雜；長視窗較穩定，卻容易過時。若依驗證期夏普比率選擇長度，就等於用配置方法受評估的同一段路徑來擬合它，無法區分改善究竟是真實效果還是擬合造成的。回溯期間取自 `config/setup.yaml`，理由與風險覆蓋層參數相同。

### 為何排除等權重，而非在此重新執行
<!-- ML4T_ZH_END c0001-p001 -->

Equal weight is the baseline, and the paragraph above the parameter cell records what happens
if it is run again in this stage: because `stage` is not part of `backtest_hash`, the row
hashes identically to its baseline parent and one of the two is silently lost. Measured in
this case study's own pre-rebuild store, 48 rows carried `stage='signal'` while declaring
`allocation.method='equal_weight'`, and there were no allocation-stage equal-weight rows at
all. The comparison a reader wants - allocator against equal weight - is made against the
baseline population, not by recomputing it here.

### These candidates stay eligible

Allocation results are part of the final selection pool, so an allocator that genuinely
improves validation Sharpe can be what the case study ships. One consequence to carry forward:
every allocator except equal weight re-sizes positions as its estimates move, which is
turnover the baseline did not have. `16_costs` is where that is priced, and an allocator's
advantage here is a gross number.

<!-- ML4T_ZH_BEGIN c0001-p002 -->
等權重就是基準，參數儲存格上方的段落記錄了在本階段重新執行的結果：由於 `stage` 不屬於 `backtest_hash`，該列與其基準父項的雜湊值相同，其中一列便悄悄遺失。本案例研究重建前的資料庫中，有 48 列帶有 `stage='signal'`，卻聲明為 `allocation.method='equal_weight'`，而配置階段的等權重資料列完全不存在。讀者要做的比較，是將配置方法與等權重相比；應與基準群體比較，而非在此重新計算等權重。

### 這些候選結果仍有入選資格

配置結果是最終選擇集合的一部分，因此，真正提高驗證期夏普比率的配置方法，可能成為本案例研究最後採用的方法。後續須記住一項影響：除等權重外，每種配置方法都會隨估計值變化而重新調整部位規模，產生基準原本沒有的周轉。`16_costs` 才會對此計價，因此此處配置方法的優勢是未扣成本的數字。
<!-- ML4T_ZH_END c0001-p002 -->


In [1]:
"""Run the declared CME futures allocation population."""

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    create_label_candidate_sets,
    open_study,
    product_universe_table,
    run_official_backtest_requests,
    shortlist_signal_configurations,
    strategy_request_frame,
)
from case_studies.research.population import supersedes_for_run
from case_studies.utils.sweep_config import get_allocators, get_top_n_predictions

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_LABELS: list[str] = []
PREVIEW_MAX_BASELINE_ROWS = 0
# None means the width `setup.yaml` declares; an int overrides it. Declared here because
# papermill only binds a name the parameters cell already holds - a run that passes
# TOP_N_PREDICTIONS to a notebook without it sweeps the declared width and exits 0.
TOP_N_PREDICTIONS = None

# The allocation population is immutable under its name, so a run whose members have moved has
# to say which generation it retires. Anything upstream that changes a backtest identity moves
# them - a corrected label, a changed accounting field, a re-run after a registry reset - and
# `OfficialPopulation.create` refuses to write a different member list under a name that already
# exists. Declared as a literal so that running the committed notebook as it stands recomputes
# the population on record. Empty for a first snapshot.
ALLOCATION_POPULATION = "cme_futures-allocation-validation-v1"
SUPERSEDES_ALLOCATION_POPULATION: str = ""

# The per-label candidate sets this notebook freezes are immutable under their names too, and
# for the same reason as the population above: `CandidateSet.create` refuses a changed member
# list under a name that already exists. Nothing reached that argument before, so any run whose
# membership moved - which a wider sweep does by construction - stopped at the freeze after the
# fit, with no parameter able to answer it.
#
# Each name maps to the generation this run retires. `"live"` names the lineage and looks the
# generation up, which is the form that does not decay: naming the head instead is correct only
# until the next publish, because `create` accepts the head and nothing else. The declaration is
# resolved through `candidate_set_supersedes` rather than offered straight, so a reader's clean
# clone - which has no generation to replace, and often no `candidate_sets` table at all -
# publishes generation one instead of being refused. An unchanged re-run never reads it: a set's
# hash is computed from its members and its contract, so the existing name binding answers.
SUPERSEDES_CANDIDATE_SETS: dict[str, str] = {
    "cme_futures-allocation-fwd_ret_5d-v1": "live",
    "cme_futures-allocation-fwd_ret_21d-v1": "live",
}

## Select signal configurations by validation Sharpe

The shortlist is deterministic. It scans the immutable signal candidate set in descending Sharpe
order with the backtest identity as tie-break, and keeps one exact checkpoint and strategy per
distinct `(family, config_name)` pair.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 依驗證期夏普比率選擇訊號設定

候選名單的產生方式是確定的。它依夏普比率遞減順序掃描不可變的訊號候選集合，以回測識別資訊作為同分時的排序依據，並為每個不同的 `(family, config_name)` 組合保留一個確切的檢查點與策略。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "canonical":
    if PREVIEW_LABELS or PREVIEW_MAX_BASELINE_ROWS:
        raise ValueError("canonical execution cannot declare preview reductions")
    labels = ALL_LABELS
elif EXECUTION_TIER == "preview":
    if WORKSPACE is None or not PREVIEW_LABELS or PREVIEW_MAX_BASELINE_ROWS < 1:
        raise ValueError(
            "preview execution requires WORKSPACE, PREVIEW_LABELS and PREVIEW_MAX_BASELINE_ROWS"
        )
    unknown = sorted(set(PREVIEW_LABELS) - set(ALL_LABELS))
    if unknown:
        raise ValueError(f"preview labels this case study does not declare: {unknown}")
    labels = tuple(PREVIEW_LABELS)
else:
    raise ValueError(f"unsupported execution tier: {EXECUTION_TIER!r}")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


## How many baseline configurations the position sizing methods run on

**Why a shortlist rather than every baseline row.** Allocation is applied to the strongest
configurations rather than to all of them, and the reason is that the question is about the
allocator, not about the configuration underneath it. Running every allocator against every
baseline row would multiply the population by the number of methods and answer a question
nobody asked, while making the effective number of trials far larger - which the deflated
Sharpe downstream then has to divide by. Keeping the strongest checkpoint and concentration
per configuration asks the allocator's question against the strategies that would otherwise
have shipped.

Canonical takes the shortlist size from `setup.yaml`, which is the declared width of the
allocation stage. A preview cannot: it backtests a bounded slice of the baseline stage, so the
canonical width names more distinct configurations than its pool contains and
`shortlist_signal_configurations` refuses - correctly, since silently returning fewer is the
quiet shrinking that strictness exists to prevent. The preview therefore declares its own
width, and is held to it just as strictly.

`TOP_N_PREDICTIONS` overrides both, which is how a sweep runs wider than the shipped
declaration. It is held to its width the same way: a number the pool cannot fill raises here
rather than quietly shrinking. A sweep that wants every configuration passes `0`, the
spelling `top_n_predictions.signal` uses one line above the declaration this overrides. That
is not the same request as a number large enough to be sure: 999 against a population of 50
is indistinguishable from a population that shrank to 50, and stays a refusal.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 部位規模決定方法會套用到多少項基準設定

**為何只選候選名單，而非每一筆基準資料列。** 配置方法會套用到最強的設定，而不是全部設定，因為此處要探討的是配置方法，不是其底層設定。讓每種配置方法都與每一筆基準資料列配對，會使群體規模乘上方法數量，回答一個沒有人提出的問題，同時大幅增加實際試驗次數，讓後續的折減夏普比率必須將其納入調整。對每項設定保留最強的檢查點與集中度，便能在原本可能採用的策略上檢驗配置方法。

正式執行從 `setup.yaml` 取得候選名單大小，這是已聲明的配置階段廣度。預覽執行不能照搬：它只回測基準階段的一個有限子集，因此正式廣度指定的不同設定數量超過預覽候選池所含數量，`shortlist_signal_configurations` 便會拒絕，這是正確的；悄悄回傳較少項目，正是嚴格檢查要防止的集合縮水。因此，預覽會聲明自己的廣度，而且同樣受到嚴格檢查。

`TOP_N_PREDICTIONS` 可覆寫兩者，讓掃描範圍大於正式發布的設定。它同樣受聲明廣度約束：候選池無法填滿指定數量時會在此報錯，而不是悄悄縮小。若要掃描每項設定，應傳入 `0`；這是 `top_n_predictions.signal` 在被覆寫的聲明上一行使用的寫法。這與給定一個看似足夠大的數字不同：若群體只有 50 項，要求 999 項的結果與群體縮小至 50 項無法區分，因此仍會遭到拒絕。
<!-- ML4T_ZH_END c0006-p000 -->


In [4]:
if TOP_N_PREDICTIONS is None:
    TOP_N_PREDICTIONS = (
        get_top_n_predictions("cme_futures", "allocation")
        if EXECUTION_TIER == "canonical"
        else PREVIEW_MAX_BASELINE_ROWS
    )
shortlist_size = TOP_N_PREDICTIONS
allocators = get_allocators("cme_futures")
if not allocators:
    raise ValueError("the configured allocator population is empty")
if any(allocation.get("method") == "equal_weight" for allocation in allocators):
    raise ValueError(
        "equal_weight is the baseline stage, not an allocator: `stage` is not part of "
        "`backtest_hash`, so an equal-weight reweight hashes identically to its baseline "
        "parent and one of the two rows is lost. Remove it from the configured menu."
    )

request_rows = []
for label in labels:
    for baseline in shortlist_signal_configurations(
        study,
        label=label,
        limit=shortlist_size,
        execution_tier=EXECUTION_TIER,
    ):
        prediction_hash = baseline.registry_record()["prediction_hash"]
        signal = baseline.spec()["strategy"]["signal"]
        for allocation in allocators:
            method = allocation["method"]
            request_rows.append(
                {
                    "request_name": f"{baseline.hash}-{method}",
                    "prediction_hash": prediction_hash,
                    "label": label,
                    "signal": signal,
                    "allocation": allocation,
                    "risk": None,
                    "costs": None,
                    "chapter": "ch17",
                }
            )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "signal", "allocation")

request_name,prediction_hash,label,signal,allocation
str,str,str,object,object
"""d29c954446e0-score_weighted""","""b467971cd825""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}",{'method': 'score_weighted'}
"""d29c954446e0-inverse_vol""","""b467971cd825""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'inverse_vol', 'vol_window': 63}"
"""d29c954446e0-risk_parity""","""b467971cd825""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'risk_parity', 'vol_window': 63}"
"""d29c954446e0-mvo_ledoit_wolf""","""b467971cd825""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'mvo_ledoit_wolf', 'lookback': 63}"
"""d29c954446e0-hrp""","""b467971cd825""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'hrp', 'vol_window': 63}"
…,…,…,…,…
"""c755d8ba3580-inverse_vol""","""0adb5f5959de""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'inverse_vol', 'vol_window': 63}"
"""c755d8ba3580-risk_parity""","""0adb5f5959de""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'risk_parity', 'vol_window': 63}"
"""c755d8ba3580-mvo_ledoit_wolf""","""0adb5f5959de""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'mvo_ledoit_wolf', 'lookback': 63}"


## Execute and freeze allocation candidates

Moment-based allocators receive only price history before each decision. Product-keyed typed
decisions retain the selected prediction, roll audit, expiry reference, and allocation settings.

"Only price history before each decision" is the causality condition for this stage, and it is
the one an allocator makes easy to violate: a covariance or volatility estimate computed over
the whole sample would size every position using dispersion the market had not yet shown, and
the resulting book would look well-balanced for reasons unavailable at the time. The estimate
is rebuilt at each decision from history strictly before it.

The results freeze as a named population, and `SUPERSEDES_ALLOCATION_POPULATION` in the
parameter cell names the generation a re-run retires. The retired snapshot stays in the
registry, so a Sharpe quoted from it remains traceable to the set it was computed over.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 執行並封存配置候選結果

以動差估計為基礎的配置方法，在每次決策時只能取得決策前的價格歷史。以商品為鍵值、具有型別的決策，會保留選定的預測、轉倉稽核、到期日參照及配置設定。

「每次決策前的價格歷史」是本階段的因果性條件，也最容易被配置方法違反：若用整個樣本計算共變異數或波動率估計值，就會利用市場當時尚未呈現的離散程度來決定每個部位的規模，使投資組合看似平衡，原因卻是當時無從得知的資訊。因此，每次決策都只使用嚴格早於該時點的歷史資料重新估計。

結果會封存為具名群體；參數儲存格中的 `SUPERSEDES_ALLOCATION_POPULATION` 指定重新執行時要汰換的版本。已汰換的快照仍留在登錄資料庫，因此引用其夏普比率時，仍可追溯到計算所依據的集合。
<!-- ML4T_ZH_END c0008-p000 -->


In [5]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name=ALLOCATION_POPULATION if EXECUTION_TIER == "canonical" else None,
    supersedes=supersedes_for_run(
        study,
        population_name=ALLOCATION_POPULATION,
        declared=SUPERSEDES_ALLOCATION_POPULATION or None,
        execution_tier=EXECUTION_TIER,
    ),
)
candidate_sets = (
    create_label_candidate_sets(
        study, execution, stage="allocation", supersedes_by_set=SUPERSEDES_CANDIDATE_SETS
    )
    if EXECUTION_TIER == "canonical"
    else {}
)

case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


... (10 outputs left out)

case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


case_studies/utils/conformal.py:515: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


`source` says whether each member was computed by this run or served from the registry because
an identical identity was already recorded. A re-run of a registered sweep is entirely `reused`
and completes in seconds; without the column that is indistinguishable from having computed
every row.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
`source` 表示每個成員是本次執行新計算的，還是因為相同識別資訊已記錄在登錄資料庫，而直接取用既有結果。重新執行已登錄的掃描時，所有成員都會是 `reused`，幾秒內即可完成；若沒有這一欄，就無法區分它與重新計算了每一列的情況。
<!-- ML4T_ZH_END c0010-p000 -->


In [6]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete,source
str,str,str,str,str,bool,str
"""221adc6dd45c-conformal_weighte…","""fwd_ret_21d""","""e5d8070db141""","""4316a07f4119""","""30dc498d6ba3""",true,"""computed"""
"""221adc6dd45c-hrp""","""fwd_ret_21d""","""e5d8070db141""","""af8ab8e8cd2b""","""a435ae8c78c9""",true,"""computed"""
"""221adc6dd45c-inverse_vol""","""fwd_ret_21d""","""e5d8070db141""","""f64325f41c07""","""aa1cc0d24246""",true,"""computed"""
"""221adc6dd45c-mvo_ledoit_wolf""","""fwd_ret_21d""","""e5d8070db141""","""7c5351b8f7ae""","""307a88c60fe8""",true,"""computed"""
"""221adc6dd45c-risk_parity""","""fwd_ret_21d""","""e5d8070db141""","""11f10e4c91f1""","""af0cd7c3210c""",true,"""computed"""
…,…,…,…,…,…,…
"""fa9a216a0f57-hrp""","""fwd_ret_5d""","""5bf0fbed0905""","""a199e914bb64""","""ba49539ae610""",true,"""computed"""
"""fa9a216a0f57-inverse_vol""","""fwd_ret_5d""","""5bf0fbed0905""","""88ce20ed1120""","""7272a330ffa7""",true,"""computed"""
"""fa9a216a0f57-mvo_ledoit_wolf""","""fwd_ret_5d""","""5bf0fbed0905""","""9f72a67e4209""","""dcd562cf83b6""",true,"""computed"""


The next two execution notebooks select the highest validation Sharpe from the union of signal and
allocation results for each label. Cost sensitivity is diagnostic; risk overlays remain eligible
for final selection.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
接下來兩份執行用 Notebook，會針對每個標籤，從訊號與配置結果的聯集中選出驗證期夏普比率最高者。成本敏感度分析僅供診斷；風險覆蓋層仍有最終入選資格。
<!-- ML4T_ZH_END c0012-p000 -->
